In [1]:
import json, shutil, os, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from threadpoolctl import threadpool_limits
threadpool_limits(limits=1)
from sklearn.metrics import roc_auc_score, average_precision_score

In [2]:
TASK = Path.cwd().parent
INPUT, OUTPUT = TASK / 'input', TASK / 'output'

In [3]:
config = json.loads((INPUT / 'config.json').read_text())
SEED = config['seed']
rng = np.random.default_rng(SEED)

In [4]:
for cell in json.loads((INPUT / 'helpers.ipynb').read_text())['cells']:
    exec(''.join(cell['source']), globals())
carry(INPUT, OUTPUT)

In [5]:
index = pd.read_csv(INPUT / 'prediction_index.csv')
truth = np.load(INPUT / 'truth.npz')['ancestry']
with np.load(INPUT / 'predictions.npz') as data:
    pred = {key: data[key] for key in data.files}

In [6]:
trial_rows, open_rows = [], []
for record in index.to_dict('records'):
    q, p = record['query_id'], record['prediction_id']
    target, path, probabilities = truth[q], pred['paths'][p], pred['probabilities'][p]
    trial_rows.append({**record, **score_path(target, path, probabilities)})
trials = pd.DataFrame(trial_rows)
trials.to_csv(OUTPUT / 'trial_metrics.csv', index=False)
keys = ['panel', 'markers', 'steps', 'kind', 'variant', 'method']
summary = trials.groupby(keys, as_index=False).agg(
    n=('mosaic_id', 'nunique'), block_accuracy=('block_accuracy', 'mean'), boundary_f1=('boundary_f1', 'mean'),
    tolerant_boundary_f1=('boundary_f1_tolerant', 'mean'), known_nll=('known_nll', 'mean'), known_brier=('known_brier', 'mean'),
    predicted_switches=('predicted_switches', 'mean'), true_switches=('true_switches', 'mean'), digit_accuracy=('digit_accuracy', 'mean'))
summary.to_csv(OUTPUT / 'summary.csv', index=False)

In [7]:
# paired resampling by complete mosaic; blocks are dependent
paired_rows = []
known = trials.query("kind in ['long', 'short', 'single'] and variant == 'aligned'")
for keys, group in known.groupby(['panel', 'markers', 'steps', 'kind']):
    pivot = group.pivot(index='mosaic_id', columns='method', values='block_accuracy')
    delta = (pivot.copying - pivot.nearest).to_numpy()
    samples = rng.choice(delta, (config['bootstrap_replicates'], len(delta)), replace=True).mean(axis=1)
    paired_rows.append(dict(zip(['panel', 'markers', 'steps', 'kind'], keys),
        mean_gain=float(delta.mean()), ci_low=float(np.quantile(samples, 0.025)), ci_high=float(np.quantile(samples, 0.975)),
        n_mosaics=len(delta)))
pd.DataFrame(paired_rows).to_csv(OUTPUT / 'paired_gains.csv', index=False)

In [8]:
# missing-source rejection is scored independently of forced closed-set calls
for keys, group in index.query("variant == 'aligned' and kind in ['long', 'missing'] and method == 'copying'").groupby(['panel', 'markers', 'steps']):
    ids, queries = group.prediction_id.to_numpy(), group.query_id.to_numpy()
    target = (truth[queries] == 3).ravel()
    scores, reject = pred['unknown_score'][ids].ravel(), pred['rejected'][ids].ravel()
    open_rows.append(dict(zip(['panel', 'markers', 'steps'], keys),
        unknown_auroc=float(roc_auc_score(target, scores)), unknown_average_precision=float(average_precision_score(target, scores)),
        unknown_recall=float(reject[target].mean()), known_false_rejection=float(reject[~target].mean()),
        unknown_blocks=int(target.sum()), known_blocks=int((~target).sum())))
pd.DataFrame(open_rows).to_csv(OUTPUT / 'open_set_metrics.csv', index=False)

In [9]:
# abstention curves use posterior marginals; not guarantees of calibration
risk_rows = []
for keys, group in index.query("variant == 'aligned' and kind == 'long' and method == 'copying'").groupby(['panel', 'markers', 'steps']):
    ids, queries = group.prediction_id.to_numpy(), group.query_id.to_numpy()
    probability = pred['probabilities'][ids]
    confidence = probability.max(axis=2).ravel()
    correct = (probability.argmax(axis=2) == truth[queries]).ravel()
    for cutoff in [0, 0.5, 0.7, 0.9, 0.95, 0.99]:
        keep = confidence >= cutoff
        risk_rows.append(dict(zip(['panel', 'markers', 'steps'], keys), threshold=cutoff,
            coverage=float(keep.mean()), accuracy=float(correct[keep].mean()) if keep.any() else np.nan))
pd.DataFrame(risk_rows).to_csv(OUTPUT / 'risk_coverage.csv', index=False)
summary.query("markers == 4 and steps == 300 and variant == 'aligned' and kind == 'long'")

,panel,markers,steps,kind,variant,method,n,block_accuracy,boundary_f1,tolerant_boundary_f1,known_nll,known_brier,predicted_switches,true_switches,digit_accuracy
102,exact,4,300,long,aligned,copying,24,0.517361,0.318913,0.519742,1.022441,0.610736,10.958333,5.458333,0.973241
103,exact,4,300,long,aligned,global,24,0.286458,0.000000,0.000000,4.200934,1.365636,0.000000,5.458333,0.973241
104,exact,4,300,long,aligned,nearest,24,0.505208,0.340093,0.491408,1.024869,0.612404,13.208333,5.458333,0.973241
402,proxy,4,300,long,aligned,copying,24,0.434028,0.297100,0.395180,1.069186,0.645020,11.083333,5.458333,0.973241
403,proxy,4,300,long,aligned,global,24,0.286458,0.000000,0.000000,6.035932,1.409271,0.000000,5.458333,0.973241
404,proxy,4,300,long,aligned,nearest,24,0.434028,0.297100,0.395180,1.069186,0.645020,11.083333,5.458333,0.973241
